# 01 抓取 Statcast 逐球数据

## 这一步做了什么
用 pybaseball.statcast() 从 Baseball Savant 下载指定日期全部比赛的逐球数据，
筛选出后续分析需要的列，按日期保存为 CSV。

## 数据从哪来
- 数据源：MLB Baseball Savant，通过 pybaseball 封装
- 粒度：一次投球一行
- 原始数据 119 列，保留 28 列；姓名映射后另加 2 列，最终保存 30 列

## 结果是什么
- 输出：data/raw/2025-06-14_statcast.csv
- 规模：约 4400 行、15 场比赛
- 缺失说明：events 只在打席终结球有值；launch_speed / launch_angle 只在球被击出时有值

## 为什么保留这些列
- 7 个基础列用于基本统计
- game_pk / inning / inning_topbot / outs_when_up / on_1b~on_3b 用于构造 24 种局面（Day 3）
- bat_score / post_bat_score 用于计算半局得分，Day 3 的得分期望必须有这两列


In [1]:
from pathlib import Path
import time

import pandas as pd
import pybaseball as pb
from pybaseball import cache

cache.enable()


def find_project_root(marker="requirements.txt") -> Path:
    """向上找含 requirements.txt 的目录，避免依赖当前工作目录。"""
    p = Path.cwd().resolve()
    for cand in [p, *p.parents]:
        if (cand / marker).exists():
            return cand
    return p


ROOT = find_project_root()
RAW_DIR = ROOT / "data" / "raw"
RAW_DIR.mkdir(parents=True, exist_ok=True)

print("项目根目录:", ROOT)
print("pybaseball:", pb.__version__, "| pandas:", pd.__version__)


项目根目录: C:\Users\GOOD\PycharmProjects\Baseball-Analytics
pybaseball: 2.2.7 | pandas: 3.1.1


In [2]:
def fetch_statcast_day(date_str: str, max_retry: int = 3, wait: int = 10) -> pd.DataFrame:
    """抓取单日全部比赛的逐球数据，失败时递增等待后重试。date_str 格式 'YYYY-MM-DD'。"""
    for attempt in range(1, max_retry + 1):
        try:
            df = pb.statcast(start_dt=date_str, end_dt=date_str, verbose=True)
            print(f"抓取成功: {df.shape}")
            return df
        except Exception as e:
            print(f"[第 {attempt} 次失败] {type(e).__name__}: {e}")
            if attempt < max_retry:
                time.sleep(wait * attempt)

    raise RuntimeError(
        f"{date_str} 抓取失败。先检查网络或代理；确认该日期有比赛；"
        f"仍不行就用末尾的手动 CSV 方案。"
    )


In [3]:
DATE = "2025-06-14"

raw = fetch_statcast_day(DATE)

raw[["game_date", "pitcher", "batter", "release_speed",
     "events", "launch_speed", "launch_angle"]].head()


This is a large query, it may take a moment to complete


100%|██████████| 1/1 [00:00<00:00,  7.70it/s]

抓取成功: (4435, 119)


,game_date,pitcher,batter,release_speed,events,launch_speed,launch_angle
2426,2025-06-14,669276,687597,87.9,strikeout,<NA>,<NA>
2546,2025-06-14,669276,687597,87.9,NaN,<NA>,<NA>
2578,2025-06-14,669276,687597,86.3,NaN,<NA>,<NA>
2700,2025-06-14,669276,687597,87.0,NaN,<NA>,<NA>
2751,2025-06-14,669276,687597,87.8,NaN,<NA>,<NA>


In [4]:
KEEP_COLS = [
    # 需求指定的 7 列
    "game_date", "pitcher", "batter", "release_speed", "events",
    "launch_speed", "launch_angle",
    # 比赛定位
    "game_pk", "game_type", "home_team", "away_team",
    # 打席上下文，Day 3 构造局面要用
    "inning", "inning_topbot", "outs_when_up", "on_1b", "on_2b", "on_3b",
    "at_bat_number", "pitch_number", "description", "type", "bb_type",
    "stand", "p_throws",
    # 得分列，Day 3 计算得分期望必须有
    "bat_score", "post_bat_score", "post_home_score", "post_away_score",
]

cols = [c for c in KEEP_COLS if c in raw.columns]
missing = [c for c in KEEP_COLS if c not in raw.columns]
if missing:
    print("数据中不存在的列，已跳过:", missing)

df = raw[cols].copy()
df["game_date"] = pd.to_datetime(df["game_date"]).dt.strftime("%Y-%m-%d")

out_path = RAW_DIR / f"{DATE}_statcast.csv"
df.to_csv(out_path, index=False, encoding="utf-8-sig")

print("已保存:", out_path)
print("形状:", df.shape, "| 大小:", f"{out_path.stat().st_size / 1024:.0f} KB")


已保存: C:\Users\GOOD\PycharmProjects\Baseball-Analytics\data\raw\2025-06-14_statcast.csv
形状: (4435, 28) | 大小: 436 KB


In [7]:
REQUIRED = [
    "game_date", "pitcher", "batter", "release_speed", "events",
    "launch_speed", "launch_angle", "game_pk", "inning", "inning_topbot",
    "outs_when_up", "on_1b", "on_2b", "on_3b", "at_bat_number",
    "pitch_number", "bat_score", "post_bat_score",
]
absent = [c for c in REQUIRED if c not in df.columns]
assert not absent, f"缺少必需列: {absent}"

print("行数:", len(df), "| 比赛场数:", df["game_pk"].nunique())
print(f"events 非空: {df['events'].notna().sum()} ({df['events'].notna().mean():.1%})")
print("\n得分列非空率:")
print(df[["bat_score", "post_bat_score"]].notna().mean().round(3).to_string())
print("\n缺失率（%）:")
print((df.isna().mean().sort_values(ascending=False) * 100).round(1).to_string())
print("\nevents 分布（前 12）:")
print(df["events"].value_counts(dropna=False).head(12).to_string())


行数: 4435 | 比赛场数: 15
events 非空: 1140 (25.7%)

得分列非空率:
bat_score         1.0
post_bat_score    1.0

缺失率（%）:
on_3b              90.1
bb_type            83.2
on_2b              80.2
events             74.3
on_1b              70.0
launch_speed       67.0
launch_angle       66.9
release_speed       0.7
game_type           0.0
home_team           0.0
batter              0.0
game_pk             0.0
pitcher             0.0
game_date           0.0
outs_when_up        0.0
inning_topbot       0.0
inning              0.0
away_team           0.0
at_bat_number       0.0
pitch_number        0.0
description         0.0
type                0.0
stand               0.0
p_throws            0.0
bat_score           0.0
post_bat_score      0.0
post_home_score     0.0
post_away_score     0.0
pitcher_name        0.0
batter_name         0.0

events 分布（前 12）:
events
NaN                          3295
field_out                     462
strikeout                     262
single                        148
walk         

In [8]:
# 把 MLBAM ID 映射成姓名。首次调用会下载球员注册表（约 10MB），之后走缓存。
try:
    from pybaseball import playerid_reverse_lookup

    ids = pd.concat([df["pitcher"], df["batter"]]).dropna().astype(int).unique()
    lookup = playerid_reverse_lookup(ids, key_type="mlbam")
    lookup["name"] = (lookup["name_first"].str.title() + " "
                      + lookup["name_last"].str.title())
    name_map = dict(zip(lookup["key_mlbam"], lookup["name"]))

    df["pitcher_name"] = df["pitcher"].map(name_map)
    df["batter_name"] = df["batter"].map(name_map)
    df.to_csv(out_path, index=False, encoding="utf-8-sig")
    print(df[["pitcher_name", "batter_name", "release_speed"]].head().to_string())

except Exception as e:
    print("姓名映射跳过，不影响后续分析:", type(e).__name__, e)


     pitcher_name  batter_name  release_speed
2426    Dylan Lee  Jordan Beck           87.9
2546    Dylan Lee  Jordan Beck           87.9
2578    Dylan Lee  Jordan Beck           86.3
2700    Dylan Lee  Jordan Beck           87.0
2751    Dylan Lee  Jordan Beck           87.8


In [9]:
# 网络不可用时：在 Baseball Savant 网页导出 CSV，放到项目根目录，用这个函数读入，
# 然后重跑单元格 5 之后的代码（把 raw 换成这里的返回值）。
def load_manual_savant_csv(path: str) -> pd.DataFrame:
    d = pd.read_csv(path, low_memory=False)
    d["game_date"] = pd.to_datetime(d["game_date"]).dt.strftime("%Y-%m-%d")
    return d


# raw = load_manual_savant_csv(ROOT / "statcast_2025-06-14.csv")
